# Reveil：在 Colab A100 上訓練馬賽克還原 IC-LoRA

這本 notebook 在同一個 A100 執行階段跑完整流程：

1. 檢查環境、掛載 Google Drive
2. 安裝 Reveil 和 Lightricks 官方訓練器 `ltx-trainer`
3. 下載 LTX-2.5 模型
4. 對齊同片名的有碼版與無碼版，偵測真正的馬賽克，產生成對訓練資料
5. 用官方前處理把配對編碼成 latent
6. 訓練 IC-LoRA：每 250 步存檔到 Drive，斷線後從頭重跑整本 notebook 會自動續訓

**開始前**

- 執行階段 → 變更執行階段類型 → **A100 GPU**（建議開高 RAM）。
- 到 [Lightricks/LTX-2.5](https://huggingface.co/Lightricks/LTX-2.5) 同意授權條款，建立 Read token，存成 Colab 密鑰 `HF_TOKEN`（左側鑰匙圖示）。
- 有碼正式發行版放在 `reveil/sources/censored/<片名>/<影片>`，同片的無碼版放在 `reveil/sources/uncensored/<片名>/<影片>`。兩側每個片名資料夾各放一部影片，可以是 Drive 捷徑；**依片名資料夾配對，影片檔名可以不同**。同一檔案放在兩側、缺少其中一側或有多部候選影片時，會警告並略過。
- 預設 `TARGET = "leak"`、`COLOUR_FIT = False`：輸入是有碼裁切片段，訓練目標是相同影格、相同裁切框的無碼原始片段，不做校色或混合。`TARGET = "composite"` 仍可選用，以有碼畫面為底，只在馬賽克區域混入對齊後的無碼畫面並平滑邊緣。配對要涵蓋不同演員和場景，太少會讓模型只背下那幾段。
- 每個視窗為 121 格、512×512 或 768×768；`RESOLUTIONS = "512,768"` 會依配對原有尺寸分開前處理，再一起訓練，改成 `"512"` 則全部以 512 訓練（768 片段會縮小）。正式資料每個片名最多 `MAX_WINDOWS_PER_TITLE = 400` 個視窗；`KEEP_LAST_N = 0` 保留每一個 checkpoint。
- 實測三個 CPU 階段合計每秒影片約需 **2.5 CPU 秒**：每部約 2.3 小時、1080p 的片名約需 **5 CPU 小時**；目前 15 個片名合計約 35 小時影片，約需 **90 CPU 小時**。在 A100 執行階段（12 個 vCPU）約需 **8–10 小時**，GPU 閒置仍計費，建議先在其他電腦做好資料。另需模型下載、latent 前處理和訓練時間；三個階段都會跳過已完成的片名。
- 素材只能用合法取得、所有演出者都確認成年的影片。

透過 Colab CLI 執行時，請先將 Drive 掛載至 `/content/drive/MyDrive`，並將 Hugging Face token 放在 `~/.cache/huggingface/token`。
使用 `colab exec -s SESSION -f notebooks/train_colab.ipynb --env REVEIL_HEADLESS=1` 啟用無介面模式，此時不讀取 Colab 密鑰，也不呼叫 `drive.mount`。
可透過 `--env` 覆寫設定：`REVEIL_SMOKE_TEST`、`REVEIL_REBUILD_DATA`、`REVEIL_COLOUR_FIT` 使用 `0`／`1`，`REVEIL_STEPS`、`REVEIL_MAX_WINDOWS_PER_TITLE`、`REVEIL_KEEP_LAST_N` 使用整數，`REVEIL_RUN_NAME` 指定訓練名稱，`REVEIL_TARGET` 指定 `leak` 或 `composite`，`REVEIL_RESOLUTIONS` 指定以逗號分隔的畫布邊長（預設 `512,768`）。
`REVEIL_REAL_SOURCES` 可指定其他素材掛載路徑（例如 `/content/src`），其中須包含 `censored/` 與 `uncensored/`，Drive 以外的素材資料夾只檢查是否存在，不會建立。

**`SMOKE_TEST` 預設是 `True`**：在選取範圍內挑兩側檔案大小合計最小的片名，跑真正的對齊和區域偵測，最多產生 1 個視窗、訓練 20 步；對齊和區域檔留在 Drive，正式訓練可沿用。它仍要解碼整部影片，首次不一定很快；真實配對模式只列數字統計，不顯示預覽影像。跑通後把 `SMOKE_TEST` 改成 `False` 正式訓練。

`DATA_MODE = "synthetic"` 可切回舊流程：只讀無碼原始影片，自動標註後加上假馬賽克（冒煙測試用隨機區域），再訓練還原。

In [ ]:
# ---- 設定 ----
import os

# 由 colab exec --env REVEIL_HEADLESS=1 設定；此模式不讀 Colab 密鑰，也不呼叫 drive.mount。
HEADLESS = os.environ.get("REVEIL_HEADLESS") == "1"
DATA_MODE = "real"  # "real"：有碼／無碼真實配對；"synthetic"：無碼影片加假馬賽克
SMOKE_TEST = True  # 真實配對取最小片名、最多 1 個視窗，訓練 20 步；正式訓練改成 False
STEPS = 3000  # 正式訓練步數
KEEP_LAST_N = 0  # 0：保留每一個 checkpoint；正整數：只留最近幾個
REBUILD_DATA = False  # True：新增成對素材、無碼影片或標註後，增量產生配對並前處理
RUN_NAME = "ic_lora_512_real_v1"
CAPTION = "A sharp, detailed, high quality video."  # 所有配對共用；改了要刪掉 Drive 上的 precomputed/
WORKERS = 8  # 同時處理幾個片名；真實配對時依片名數和 CPU 核心數調低
MIN_SAMPLES = 100  # 正式訓練最少要有幾個配對，不到就不訓練

DRIVE_ROOT = "/content/drive/MyDrive/reveil"  # 跨 session 保存的東西都在這裡
LOCAL = "/content/work"

# 真實配對設定（DATA_MODE == "real"）
REAL_SOURCES = f"{DRIVE_ROOT}/sources"  # censored/<片名>/ 與 uncensored/<片名>/ 各一部影片
ALIGN = f"{DRIVE_ROOT}/align"
REGIONS_REAL = f"{DRIVE_ROOT}/regions/real"
TARGET = "leak"  # "leak"：相同影格、相同框的無碼原始裁切；"composite"：只替換馬賽克區域
COLOUR_FIT = False  # True：將無碼片段的色彩調整至與有碼片段一致
MAX_WINDOWS_PER_TITLE = 400
RESOLUTIONS = "512,768"  # 以逗號分隔的訓練畫布邊長；"512"：全部以 512 訓練，768 片段會縮小
ONLY_TITLES = []  # 空 = 全部片名；填片名資料夾名稱
SIG_CACHE = f"{LOCAL}/sigcache"

# 以下設定只適用於 DATA_MODE == "synthetic"
MAX_WINDOWS_PER_VIDEO = 100  # 每部影片最多取幾組配對（一組約 4–5 秒），讓資料平均來自不同影片；None = 不限
# 偵測要打碼的部位。NudeNet（AGPL-3.0）分得出生殖器、肛門等 18 類；一般影片肛門不打碼，所以只取生殖器。
# 也可以改用 "lada_nsfw_detection_model_v1.3"（只有一類 nsfw，會連肛門一起標；三個類別設定都要改成 []）。
DETECTOR = "nudenet_640m"
DETECT_CLASSES = ["FEMALE_GENITALIA_EXPOSED", "MALE_GENITALIA_EXPOSED"]  # 空 = 模型的全部類別
DETECT_EXCLUDE = ["ANUS_EXPOSED"]  # 從生殖器的框裡挖掉，不打碼
DETECT_DOUBT = ["FACE_FEMALE", "FACE_MALE"]  # 框大半落在臉裡的不算（例如舌吻的嘴被誤判）
DETECT_CONF = 0.25  # 每一格的門檻；放低一點，框才不會斷斷續續
DETECT_TRACK_SCORE = 0.4  # 一整段的分數中位數低於這個就不打碼（例如手被誤判），預覽上畫成紅框
# 訓練用的無碼影片資料夾，可以列好幾個；子資料夾和 Drive 捷徑都會讀。有碼片（sources/ 裡的 AV）不能放進來。
SOURCES = [f"{DRIVE_ROOT}/sources/FC2"]
REGIONS = f"{DRIVE_ROOT}/regions/{DETECTOR}"  # 每個偵測模型各一個資料夾：換模型就重新標註
PREVIEWS = f"{DRIVE_ROOT}/previews/{DETECTOR}"

REVEIL_REPO = "https://github.com/7WayneLee/Reveil"
REVEIL_REF = "claude/real-pairs"  # 合併到 main 後改成 "main"
LTX2_REF = "e98f6d927d83c92ef2be6aec76c8440b19306e39"  # 這本 notebook 對照過的 LTX-2 commit

# CLI 自動執行時可用環境變數覆寫；未設定時保留上方預設值。
overrides = []
if "REVEIL_SMOKE_TEST" in os.environ:
    SMOKE_TEST = os.environ["REVEIL_SMOKE_TEST"] == "1"
    overrides.append(f"SMOKE_TEST={SMOKE_TEST}")
if "REVEIL_STEPS" in os.environ:
    STEPS = int(os.environ["REVEIL_STEPS"])
    overrides.append(f"STEPS={STEPS}")
if "REVEIL_KEEP_LAST_N" in os.environ:
    KEEP_LAST_N = int(os.environ["REVEIL_KEEP_LAST_N"])
    overrides.append(f"KEEP_LAST_N={KEEP_LAST_N}")
if "REVEIL_RUN_NAME" in os.environ:
    RUN_NAME = os.environ["REVEIL_RUN_NAME"]
    overrides.append(f"RUN_NAME={RUN_NAME}")
if "REVEIL_MAX_WINDOWS_PER_TITLE" in os.environ:
    MAX_WINDOWS_PER_TITLE = int(os.environ["REVEIL_MAX_WINDOWS_PER_TITLE"])
    overrides.append(f"MAX_WINDOWS_PER_TITLE={MAX_WINDOWS_PER_TITLE}")
if "REVEIL_REBUILD_DATA" in os.environ:
    REBUILD_DATA = os.environ["REVEIL_REBUILD_DATA"] == "1"
    overrides.append(f"REBUILD_DATA={REBUILD_DATA}")
if "REVEIL_REAL_SOURCES" in os.environ:
    REAL_SOURCES = os.environ["REVEIL_REAL_SOURCES"]
    overrides.append(f"REAL_SOURCES={REAL_SOURCES}")
if "REVEIL_TARGET" in os.environ:
    TARGET = os.environ["REVEIL_TARGET"]
    overrides.append(f"TARGET={TARGET}")
if "REVEIL_COLOUR_FIT" in os.environ:
    COLOUR_FIT = os.environ["REVEIL_COLOUR_FIT"] == "1"
    overrides.append(f"COLOUR_FIT={COLOUR_FIT}")
if "REVEIL_RESOLUTIONS" in os.environ:
    RESOLUTIONS = os.environ["REVEIL_RESOLUTIONS"]
    overrides.append(f"RESOLUTIONS={RESOLUTIONS}")
if overrides:
    print("環境變數覆寫：" + ", ".join(overrides))

KIND = "smoke" if SMOKE_TEST else "main"
MODELS = "/content/models/ltx-2.5"
DATA = f"{LOCAL}/data/{DATA_MODE}_{KIND}"  # 配對、dataset.jsonl、.precomputed（本機 SSD）
PRECOMPUTED = f"{DATA}/.precomputed"
DRIVE_PAIRS = f"{DRIVE_ROOT}/pairs/{DATA_MODE}_{KIND}"
DRIVE_PRECOMPUTED = f"{DRIVE_ROOT}/precomputed/{DATA_MODE}_{KIND}"
RUN_DIR = f"{DRIVE_ROOT}/runs/{RUN_NAME}" + ("-smoke" if SMOKE_TEST else "")
TRAINER = "/content/LTX-2/packages/ltx-trainer"
SHARED_CONDITION = "shared_condition.pt"
if DATA_MODE not in ("real", "synthetic"):
    raise ValueError('DATA_MODE 只能是 "real" 或 "synthetic"')
if WORKERS < 1:
    raise ValueError("WORKERS 必須大於 0")

## 1. 環境檢查

In [ ]:
import os
import shlex
import shutil
import signal
import subprocess
import sys
import threading
import time
from collections import deque
from pathlib import Path


def sh(cmd, cwd=None, show=None):
    # Run a command (list or string), streaming its output; stop on failure with its last lines.
    if isinstance(cmd, str):
        cmd = shlex.split(cmd)
    cmd = [str(part) for part in cmd]
    print("$", show or shlex.join(cmd), flush=True)
    # Colab's inline matplotlib backend exists only inside this kernel, and Colab's PYTHONPATH
    # must not leak into the LTX-2 virtualenv.
    env = dict(os.environ, MPLBACKEND="Agg")
    if cmd[0] == "uv":
        env.pop("PYTHONPATH", None)
    # Its own process group: whatever the command starts (worker processes, the trainer under
    # uv) is stopped with it when it ends, fails or the cell is interrupted.
    process = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1, start_new_session=True)
    last_lines = deque(maxlen=5)

    def pump():  # in a thread: processes left behind may hold the pipe open after the command exits
        for line in process.stdout:
            print(line, end="", flush=True)
            last_lines.append(line.rstrip())

    reader = threading.Thread(target=pump, daemon=True)
    reader.start()
    try:
        while process.poll() is None:  # short sleeps, so pressing stop interrupts at once
            time.sleep(0.2)
    finally:
        try:  # stop whatever the command left running: SIGTERM, then SIGKILL after 5 s
            os.killpg(process.pid, signal.SIGTERM)
            for _ in range(25):
                time.sleep(0.2)
                os.killpg(process.pid, 0)
            os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        process.wait()
        reader.join()
    if process.returncode:
        raise RuntimeError(f"command failed with exit code {process.returncode}:\n" + "\n".join(last_lines))


def sync(src, dst, exclude=()):
    # Copy files that are new or changed (size or mtime) from src to dst; skip symlinks and
    # the top-level names in `exclude`.
    copied = 0
    for root, dirs, files in os.walk(src):
        rel = os.path.relpath(root, src)
        if rel == ".":
            dirs[:] = [d for d in dirs if d not in exclude]
            files = [f for f in files if f not in exclude]
        os.makedirs(os.path.join(dst, rel), exist_ok=True)
        for name in files:
            source, target = os.path.join(root, name), os.path.join(dst, rel, name)
            if os.path.islink(source):
                continue
            if (not os.path.exists(target) or os.path.getsize(target) != os.path.getsize(source)
                    or os.path.getmtime(source) > os.path.getmtime(target) + 1):
                shutil.copy2(source, target)
                copied += 1
    print(f"sync {src} -> {dst}: {copied} files copied", flush=True)


def readable_videos(folders):
    # The videos under `folders`, each checked to open and read, so a broken Drive mount stops
    # the run here instead of failing video by video in the long steps.
    from reveil.make_pairs import find_videos

    missing = [folder for folder in folders if not os.path.isdir(folder)]
    if missing:
        raise RuntimeError(f"找不到 {missing}：檢查 SOURCES，或在 Drive 上建好捷徑")
    videos = find_videos([Path(folder) for folder in folders])
    unreadable = []
    for video in videos:
        try:
            with open(video, "rb") as f:
                f.read(1024)
        except OSError as error:
            unreadable.append(f"{video.name}（{error.strerror}）")
    if unreadable:
        raise RuntimeError(f"{len(unreadable)}/{len(videos)} 部影片讀不到，例如 {unreadable[0]}。"
                           "Drive 掛載可能壞了：執行階段 → 中斷連線並刪除執行階段，再整本重跑")
    return videos


query = ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader,nounits"]
name, memory_mib, driver = (p.strip() for p in subprocess.run(query, capture_output=True, text=True, check=True).stdout.splitlines()[0].split(","))
print(f"GPU {name}, {int(memory_mib) / 1024:.0f} GiB, driver {driver}")
if int(memory_mib) < 70_000:
    print("⚠️ 不是 80GB 的卡：標註、配對、前處理照常；訓練會自動改用 INT8 量化（比較慢）")
if int(driver.split(".")[0]) < 580:
    print("⚠️ 驅動低於 580：LTX-2 用的 torch（CUDA 13.2）可能用不了 GPU")
free_gb = shutil.disk_usage("/content").free / 1e9
print(f"/content 可用 {free_gb:.0f} GB，RAM {os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9:.0f} GB，CPU {os.cpu_count()} 核")
if free_gb < 150:
    print("⚠️ 磁碟偏小：模型約 72 GB，另外要放配對和前處理結果")

In [ ]:
if os.path.ismount("/content/drive/MyDrive"):
    print("Google Drive 已掛載，沿用目前的掛載")
elif HEADLESS:
    raise RuntimeError("CLI 執行前必須先將 Google Drive 掛載至 /content/drive/MyDrive")
else:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=True)  # 重跑時重新掛載：Drive 上改過的捷徑才看得到
source_folders = ([f"{REAL_SOURCES}/censored", f"{REAL_SOURCES}/uncensored"]
                  if DATA_MODE == "real" else SOURCES)
mode_folders = [ALIGN, REGIONS_REAL, SIG_CACHE] if DATA_MODE == "real" else [REGIONS, PREVIEWS]
# 外部素材可能是唯讀掛載，只檢查是否存在。
if DATA_MODE == "real" and os.path.commonpath(
    [os.path.abspath(REAL_SOURCES), os.path.abspath(DRIVE_ROOT)]
) == os.path.abspath(DRIVE_ROOT):
    mode_folders.extend(source_folders)
for path in (*mode_folders, DRIVE_PAIRS, DRIVE_PRECOMPUTED, RUN_DIR, DATA):
    os.makedirs(path, exist_ok=True)
print(f"{DATA_MODE} 模式讀取的素材資料夾：")
for path in source_folders:
    print(("✓ " if os.path.isdir(path) else "⚠️ 找不到 ") + path)

## 2. 安裝 Reveil 和 ltx-trainer（約 5–10 分鐘）

In [ ]:
if HEADLESS:
    github_token = os.environ.get("GITHUB_TOKEN")
else:
    from google.colab import userdata

    try:
        github_token = userdata.get("GITHUB_TOKEN")  # 只有 repo 改成私有時才需要
    except Exception:
        github_token = None
url = REVEIL_REPO.replace("https://", f"https://{github_token}@") if github_token else REVEIL_REPO
shutil.rmtree("/content/Reveil", ignore_errors=True)
sh(["git", "clone", "-q", "--depth", "1", "--branch", REVEIL_REF, url, "/content/Reveil"],
   show=f"git clone {REVEIL_REPO} ({REVEIL_REF})")
sh("pip install -q av")
sh("pip install -q --no-deps -e /content/Reveil")  # numpy / opencv / pyyaml are already on Colab
if "/content/Reveil" not in sys.path:  # pip -e 只對新開的 Python 生效；這個核心直接從原始碼 import
    sys.path.insert(0, "/content/Reveil")

sh("pip install -q uv")
if not os.path.isdir("/content/LTX-2/.git"):
    sh("git init -q /content/LTX-2")
    sh("git -C /content/LTX-2 remote add origin https://github.com/Lightricks/LTX-2")
sh(["git", "-C", "/content/LTX-2", "fetch", "-q", "--depth", "1", "origin", LTX2_REF])
sh("git -C /content/LTX-2 checkout -q FETCH_HEAD")
sh("uv sync", cwd="/content/LTX-2")
sh(["uv", "run", "python", "-c", "import torch, ltx_trainer; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"],
   cwd=TRAINER)

## 3. 下載 LTX-2.5（約 72 GB，存在本機，每個 session 重新下載）

In [ ]:
sh("pip install -q -U huggingface_hub hf_xet")
if "HF_TOKEN" not in os.environ and not os.path.isfile(os.path.expanduser("~/.cache/huggingface/token")):
    if HEADLESS:
        raise RuntimeError("找不到 Hugging Face token 檔案：CLI 執行前請先放置 ~/.cache/huggingface/token，或設定 HF_TOKEN 環境變數")
    else:
        from google.colab import userdata

        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from huggingface_hub import hf_hub_download

MODEL_FILES = {
    "transformer": "diffusion_models/ltx-2.5-22b-dev-transformer-bf16.safetensors",
    "text_encoder": "text_encoders/gemma4-12b-with-proj-ltx-2.5-bf16.safetensors",
    "video_vae": "vae/ltx-2.5-video-vae-bf16.safetensors",
    "audio_vae": "vae/ltx-2.5-audio-vae-bf16.safetensors",  # 不訓練音訊，只是讓 split pack 的路徑齊全
}
MODEL_PATHS = {key: hf_hub_download("Lightricks/LTX-2.5", name, local_dir=MODELS) for key, name in MODEL_FILES.items()}
MODEL_PATHS

## 4. 訓練資料

Drive 上已經有前處理好的資料時直接還原，不重做；`REBUILD_DATA = True` 會增量處理新加入的片名或影片（做過的配對和 latent 會跳過）。配對和前處理資料依 `DATA_MODE` 與 `SMOKE_TEST` 分開保存，兩種模式不會混用。

### 在其他電腦先做好前處理（建議）

先在其他電腦依序執行與本筆記本相同的三個指令：`python -m reveil.align`、`python -m reveil.realmask`、`python -m reveil.real_pairs`。產生配對時使用相同的 `TARGET`、`COLOUR_FIT` 與 `MAX_WINDOWS_PER_TITLE`（正式資料為 400）；預設傳入 `--target leak --no-colour-fit`，並把 `{"pairs": {"caption": CAPTION}}` 寫入 YAML 檔，透過 `--config` 傳給 `reveil.real_pairs`；沿用預設種子（`--seed 0`）。

完成後將結果複製到 Drive 的 `reveil/` 資料夾：

- `align/` 與 `regions/real/`：三個階段共用的對齊與區域檔。
- `pairs/real_main/`：正式資料的 `clean/`、`mosaic/`、`meta/` 與 `dataset.jsonl`。
- `pairs/real_smoke/`：冒煙測試另以選取範圍內有碼版與無碼版檔案大小合計最小的片名，使用 `--max-windows-per-title 1` 產生配對。

三個階段都會跳過已完成的片名，因此備妥並複製完整資料後，Colab 只需執行 latent 前處理與訓練。請只在沒有任何 Colab 執行階段掛載 Drive 時複製資料。

In [ ]:
if os.path.isfile(f"{DRIVE_PRECOMPUTED}/{SHARED_CONDITION}"):
    sync(DRIVE_PRECOMPUTED, PRECOMPUTED)
    sh(["python", "-m", "reveil.conditions", "restore", PRECOMPUTED])
NEED_DATA = REBUILD_DATA or not os.path.isfile(f"{PRECOMPUTED}/{SHARED_CONDITION}")
print("需要產生 / 更新資料" if NEED_DATA else "沿用 Drive 上的前處理資料")

### 真實配對：檢查素材

僅 `DATA_MODE == "real"` 且需要建資料時執行以下四格。先列出配對和略過原因，逐一確認兩側影片能開啟並讀取 1 KB；`ONLY_TITLES` 可限制片名。冒煙測試只處理選取範圍內兩側檔案大小合計最小的一組，最多取 1 個視窗。素材可以放在 Drive 捷徑，不會複製整部影片到本機。

In [ ]:
if DATA_MODE == "real" and NEED_DATA:
    from reveil.realpairs import find_real_pairs

    real_pairs, pair_warnings = find_real_pairs(
        Path(REAL_SOURCES, "censored"), Path(REAL_SOURCES, "uncensored"))
    for warning in pair_warnings:
        print(f"⚠️ 略過：{warning}")
    if not real_pairs:
        raise RuntimeError(
            f"{REAL_SOURCES} 裡沒有可用的有碼／無碼配對："
            "請確認 censored/<片名>/ 和 uncensored/<片名>/ 各有一部可讀且不同的影片，並查看上方警告。"
        )
    unreadable = []
    for pair in real_pairs:
        print(f"{pair.title}\n  有碼：{pair.retail}\n  無碼：{pair.leak}")
        for video in (pair.retail, pair.leak):
            try:
                with open(video, "rb") as source:
                    if len(source.read(1024)) < 1024:
                        raise OSError("影片不足 1 KB")
            except OSError as error:
                unreadable.append(f"{video}（{error}）")
    if unreadable:
        raise RuntimeError(
            f"{len(unreadable)} 部影片讀不到，例如 {unreadable[0]}。"
            "Drive 掛載可能壞了：執行階段 → 中斷連線並刪除執行階段，再整本重跑。"
        )
    if ONLY_TITLES:
        missing = sorted(set(ONLY_TITLES) - {pair.title for pair in real_pairs})
        if missing:
            raise RuntimeError(f"ONLY_TITLES 中沒有可用配對的片名：{missing}；請查看上方警告。")
        real_pairs = [pair for pair in real_pairs if pair.title in ONLY_TITLES]
    if SMOKE_TEST:
        real_pairs = [min(real_pairs, key=lambda pair: pair.retail.stat().st_size + pair.leak.stat().st_size)]
        print(f"冒煙測試：只處理 {real_pairs[0].title}，最多 1 個視窗，訓練 20 步")
    # 三個階段都只讀本次找到的片名，避免 Drive 裡舊的對齊檔被一起處理。
    real_only_args = ["--only", *[pair.title for pair in real_pairs]]
    WORKERS = min(WORKERS, os.cpu_count() or 1, len(real_pairs))
    DECODE_THREADS = max(1, (os.cpu_count() or 1) // WORKERS)
    print(f"{len(real_pairs)} 組可讀配對，{WORKERS} 個處理程序，每個解碼器 {DECODE_THREADS} 個執行緒")

In [ ]:
# 對齊結果直接存到 Drive；重跑會跳過已完成的片名。
if DATA_MODE == "real" and NEED_DATA:
    sh(["python", "-m", "reveil.align", "--censored", f"{REAL_SOURCES}/censored",
        "--uncensored", f"{REAL_SOURCES}/uncensored", "--out", ALIGN, "--cache", SIG_CACHE,
        "--workers", WORKERS, "--decode-threads", DECODE_THREADS, *real_only_args])

In [ ]:
# 偵測真正的馬賽克與疊圖，只列數字統計；區域檔也直接存到 Drive。
if DATA_MODE == "real" and NEED_DATA:
    import json

    sh(["python", "-m", "reveil.realmask", "--align-dir", ALIGN, "--out", REGIONS_REAL,
        "--workers", WORKERS, "--decode-threads", DECODE_THREADS, *real_only_args])

    def median_block(histogram):
        # 區塊大小直方圖的加權中位數，不展開成逐影格陣列。
        total = sum(histogram.values())
        if not total:
            return 0.0
        middle, values, seen = ((total - 1) // 2, total // 2), [], 0
        for block, count in sorted((float(block), count) for block, count in histogram.items()):
            values.extend(block for index in middle if seen <= index < seen + count)
            seen += count
        return sum(values) / 2

    for pair in real_pairs:
        stats = json.loads(Path(REGIONS_REAL, f"{pair.title}.json").read_text())["stats"]
        alignment = json.loads(Path(ALIGN, f"{pair.title}.json").read_text())
        print(f"{pair.title}：馬賽克影格 {stats['frames_with_mosaic']}，"
              f"區塊中位數 {median_block(stats['block_histogram']):.1f} px，"
              f"疊圖 {stats['overlay_count']}，對齊比例 {alignment['stats']['matched_share']:.1%}")

In [ ]:
# 還原既有配對後增量產生，與合成模式使用相同的資料格式和同步方式。
if DATA_MODE == "real" and NEED_DATA:
    import yaml

    if os.listdir(DRIVE_PAIRS):
        sync(DRIVE_PAIRS, DATA)
    real_config = Path(LOCAL, "real_pairs.yaml")
    real_config.write_text(yaml.safe_dump({"pairs": {"caption": CAPTION}}, allow_unicode=True))
    sh(["python", "-m", "reveil.real_pairs", "--align-dir", ALIGN, "--regions-dir", REGIONS_REAL,
        "--out", DATA, "--config", real_config, "--target", TARGET,
        *([] if COLOUR_FIT else ["--no-colour-fit"]),
        "--max-windows-per-title", 1 if SMOKE_TEST else MAX_WINDOWS_PER_TITLE,
        "--workers", WORKERS, "--decode-threads", DECODE_THREADS, *real_only_args])
    sync(DATA, DRIVE_PAIRS, exclude=[".precomputed"])
    if not Path(DATA, "dataset.jsonl").read_text().strip():
        raise RuntimeError("沒有產生可訓練的真實配對：請檢查上方對齊與馬賽克統計，或增加成對素材。")

### 合成配對：自動標註與預覽（僅 synthetic 模式）

以下三格只在 `DATA_MODE == "synthetic"` 時執行，素材放在 `SOURCES`（預設 `reveil/sources/FC2/`），只能使用無碼原始影片。正式訓練、需要產生資料時才標註；冒煙測試沿用舊流程，以隨機區域、每部影片最多 1 個視窗產生配對。偵測模型找的是無碼畫面裡露出的部位（預設只取生殖器），用來決定假馬賽克打在哪裡；它不是用來找馬賽克的。第一次會下載偵測模型存到 Drive 的 `reveil/models/`，之後直接沿用（Lada 的模型是 ultralytics 權重，會先用 `yolo` 指令以獨立程式匯出成 ONNX）。開始前會先確認每部影片都讀得到。標註存在 `reveil/regions/<偵測模型>/`，預覽在 `reveil/previews/<偵測模型>/`，已經有標註檔的影片會跳過。

第一次正式訓練時，建議先選取預覽下面那一格，按 Ctrl+F8（執行階段 → 執行先前的儲存格）跑到預覽為止；確認標得準，再按 Ctrl+F10（執行之後的儲存格）繼續。預覽上綠框是會打碼的區域，紅框是偵測到但被丟掉的：分數太低（例如被誤判的手）、大半落在臉裡（例如舌吻）、或畫面暗到看不出東西。如果框太大或太小、漏標或誤標，就調整 `DETECT_CONF`、`DETECT_TRACK_SCORE`、`DETECT_CLASSES`、`DETECT_EXCLUDE`、`DETECT_DOUBT` 再重跑：設定改了，所有影片會自動重新標註（已經產生過配對的話，要先刪掉 `reveil/pairs/synthetic_main/` 和 `reveil/precomputed/synthetic_main/`）。

In [ ]:
import urllib.request

DETECTOR_URLS = {  # 依序嘗試；.onnx 直接用，.pt 是 ultralytics 權重，要先匯出
    "nudenet_640m": [  # NudeNet 官方 GitHub 的下載現在要登入，先用 Hugging Face 上的副本
        "https://huggingface.co/spaces/Nymbo/Nudity-Censor/resolve/7955acb6e51995555fe4e5f2fcde7437004053a0/640m.onnx",
        "https://huggingface.co/spaces/xxparthparekhxx/NudeNet-FastAPI/resolve/794a185a301917f1a3505ab3b8d55b268ea81f0e/640m.onnx",
        "https://github.com/notAI-tech/NudeNet/releases/download/v3.4-weights/640m.onnx",
    ],
    "lada_nsfw_detection_model_v1.3": [
        "https://huggingface.co/ladaapp/lada/resolve/main/lada_nsfw_detection_model_v1.3.pt",
    ],
}


def is_model_file(path):
    # A model file, not a login or error page that a download saved in its place.
    if not os.path.isfile(path):
        return False
    with open(path, "rb") as f:
        return not f.read(64).lstrip().startswith((b"<", b"{"))


def fetch_model(urls, path):
    for url in urls:
        try:
            urllib.request.urlretrieve(url, path)
        except OSError as error:
            print(f"下載失敗：{url}（{error}）")
            continue
        if is_model_file(path):
            print(f"下載完成（{os.path.getsize(path) / 1e6:.0f} MB）：{url}")
            return
        print(f"下載到的不是模型檔，換下一個來源：{url}")
    raise RuntimeError(f"{DETECTOR} 的模型檔都下載不到：{urls}")


if DATA_MODE == "synthetic" and NEED_DATA and not SMOKE_TEST:
    print(f"{len(readable_videos(SOURCES))} 部影片都讀得到")
    detector_onnx = f"{DRIVE_ROOT}/models/{DETECTOR}.onnx"
    if not is_model_file(detector_onnx):  # 還沒下載過，或之前存到的是錯誤頁
        urls = DETECTOR_URLS[DETECTOR]
        local = f"/content/detector/{DETECTOR}{Path(urls[0]).suffix}"
        os.makedirs(os.path.dirname(local), exist_ok=True)
        fetch_model(urls, local)
        if local.endswith(".pt"):
            sh("pip install -q ultralytics==8.4.4")  # Lada 用的版本；只用來匯出模型，以獨立程式執行
            sh(["yolo", "export", f"model={local}", "format=onnx", "dynamic=True"])
            local = str(Path(local).with_suffix(".onnx"))
        os.makedirs(os.path.dirname(detector_onnx), exist_ok=True)
        shutil.copy(local, detector_onnx)
    # 只留 GPU 版 onnxruntime：兩版同時裝會互相蓋掉檔案
    sh("pip uninstall -y -q onnxruntime onnxruntime-gpu")
    sh("pip install -q onnxruntime-gpu[cuda,cudnn]")
    annotate = ["python", "-m", "reveil.annotate", "--videos", *SOURCES, "--out", REGIONS,
                "--model", detector_onnx, "--conf", DETECT_CONF, "--workers", WORKERS,
                "--preview-dir", PREVIEWS]
    if DETECT_CLASSES:
        annotate += ["--classes", *DETECT_CLASSES]
    if DETECT_EXCLUDE:
        annotate += ["--exclude-classes", *DETECT_EXCLUDE]
    if DETECT_DOUBT:
        annotate += ["--doubt-classes", *DETECT_DOUBT]
    annotate += ["--min-track-score", DETECT_TRACK_SCORE]
    sh(annotate)

In [ ]:
if DATA_MODE == "synthetic" and NEED_DATA and not SMOKE_TEST:
    from IPython.display import Image, display

    previews = sorted(Path(PREVIEWS).glob("*.jpg"))
    print(f"{len(previews)} 張預覽，這裡顯示前 4 張，其餘在 Drive 的 {PREVIEWS}")
    for path in previews[:4]:
        print(path.name)
        display(Image(filename=str(path), width=960))

In [ ]:
if DATA_MODE == "synthetic" and NEED_DATA:
    videos = readable_videos(SOURCES)
    if not videos:
        raise RuntimeError(f"{', '.join(SOURCES)} 裡沒有影片：先把無碼原始影片放進去")
    if not SMOKE_TEST:
        missing = [v.name for v in videos if not os.path.isfile(f"{REGIONS}/{v.stem}.json")]
        if len(missing) == len(videos):
            raise RuntimeError(
                f"{len(videos)} 部影片都沒有標註檔（例如 {REGIONS}/{videos[0].stem}.json）；"
                "請看上面自動標註那格的輸出"
            )
        if missing:
            print(f"⚠️ {len(missing)} 部影片沒有標註，會被略過：{missing[:5]}")
    print(f"{len(videos)} 部影片")
    if os.listdir(DRIVE_PAIRS):
        sync(DRIVE_PAIRS, DATA)  # 已完成的影片會被跳過
    make_pairs = ["python", "-m", "reveil.make_pairs", "--videos", *SOURCES, "--out", DATA,
                  "--workers", WORKERS, "--caption", CAPTION]
    if SMOKE_TEST:
        make_pairs += ["--random-regions", "--max-windows-per-video", "1"]
    else:
        make_pairs += ["--regions", REGIONS]
        if MAX_WINDOWS_PER_VIDEO:
            make_pairs += ["--max-windows-per-video", MAX_WINDOWS_PER_VIDEO]
    sh(make_pairs)
    sync(DATA, DRIVE_PAIRS, exclude=[".precomputed"])

In [ ]:
if NEED_DATA:
    import json

    dataset = Path(DATA, "dataset.jsonl")
    rows = [json.loads(line) for line in dataset.read_text(encoding="utf-8").splitlines() if line.strip()]
    sides = sorted({int(side.strip()) for side in RESOLUTIONS.split(",")}) if DATA_MODE == "real" else [512]
    if any(side not in (512, 768) for side in sides):
        raise ValueError('RESOLUTIONS 請填 "512"、"768" 或 "512,768"')
    preprocessing = []
    if DATA_MODE == "real" and len(sides) > 1:
        # 同長寬比的 bucket 會優先選大尺寸，必須依原有畫布分開執行，避免將 512 放大成 768。
        canvas_by_video = {}
        for meta in sorted(Path(DATA, "meta").glob("*.json")):
            for window in json.loads(meta.read_text(encoding="utf-8")).get("windows", []):
                if window.get("canvas") is not None:
                    canvas_by_video[(dataset.parent / window["video"].strip()).resolve()] = int(window["canvas"])
        rows_by_side = {side: [] for side in sides}
        for row in rows:
            video = (dataset.parent / row["video"].strip()).resolve()
            canvas = canvas_by_video.get(video)
            if canvas is None:  # 舊配對沒有 canvas 時，用 PyAV 讀取第一格的尺寸。
                import av

                with av.open(str(video)) as container:
                    frame = next(container.decode(video=0))
                    width, height = frame.width, frame.height
                if width != height:
                    raise ValueError(f"{video} 不是正方形畫布：{width}×{height}")
                canvas = width
            if canvas not in rows_by_side:
                raise ValueError(f"{video} 的畫布 {canvas} 不在 RESOLUTIONS={RESOLUTIONS} 裡")
            rows_by_side[canvas].append(row)
        for side, subset_rows in rows_by_side.items():
            # 子集與完整 dataset 同一層，讓 latent 的相對路徑保持一致。
            subset = dataset.with_name(f"dataset_{side}.jsonl")
            subset.write_text("".join(json.dumps(row, ensure_ascii=False) + "\n" for row in subset_rows),
                              encoding="utf-8")
            print(f"{side}×{side}×121：{len(subset_rows)} 個配對")
            if subset_rows:
                preprocessing.append((subset, side))
    else:
        side = sides[0]
        print(f"{side}×{side}×121：{len(rows)} 個配對")
        preprocessing.append((dataset, side))
    for dataset_file, side in preprocessing:
        sh(["uv", "run", "python", "scripts/process_dataset.py", dataset_file,
            "--resolution-buckets", f"{side}x{side}x121",
            "--model-path", MODEL_PATHS["transformer"],
            "--text-encoder-path", MODEL_PATHS["text_encoder"],
            "--video-vae-path", MODEL_PATHS["video_vae"],
            "--audio-vae-path", MODEL_PATHS["audio_vae"],
            "--skip-audio", "--output-dir", PRECOMPUTED], cwd=TRAINER)
    # 每個樣本的 caption embedding 都一樣：只留一份，其餘換成連結，再存到 Drive。
    sh(["python", "-m", "reveil.conditions", "share", PRECOMPUTED, "--dataset", f"{DATA}/dataset.jsonl"])
    sync(PRECOMPUTED, DRIVE_PRECOMPUTED, exclude=["conditions"])

## 5. 訓練

checkpoint 每 250 步存到 `RUN_DIR/checkpoints/`，預設 `KEEP_LAST_N = 0` 全部保留；改成正整數則只保留最近幾個。斷線後從頭重跑整本 notebook：資料從 Drive 還原，訓練從最新的 checkpoint 接著跑。AI Pro 沒有背景執行，分頁要保持開著。

In [ ]:
import glob

import yaml

samples = sum(1 for _ in Path(PRECOMPUTED, "latents").rglob("*.pt"))
print(f"{samples} 個訓練樣本")
if not SMOKE_TEST and samples < MIN_SAMPLES:
    raise RuntimeError(
        f"只有 {samples} 個樣本（MIN_SAMPLES = {MIN_SAMPLES}），訓練 {STEPS} 步只會背下這幾段。"
        + ("增加有碼／無碼成對素材" if DATA_MODE == "real" else "多放一些無碼影片")
        + "，用 REBUILD_DATA = True 重跑；確定要這樣訓練就調低 MIN_SAMPLES。"
    )
with open("/content/Reveil/configs/ic_lora_512.yaml") as f:
    config = yaml.safe_load(f)
config["model"].update(
    model_path=MODEL_PATHS["transformer"],
    text_encoder_path=MODEL_PATHS["text_encoder"],
    video_vae_path=MODEL_PATHS["video_vae"],
    audio_vae_path=MODEL_PATHS["audio_vae"],
)
config["data"]["preprocessed_data_root"] = PRECOMPUTED
config["output_dir"] = RUN_DIR
config["optimization"]["steps"] = 20 if SMOKE_TEST else STEPS
config["checkpoints"]["keep_last_n"] = KEEP_LAST_N
if SMOKE_TEST:
    config["checkpoints"]["interval"] = 10
if int(memory_mib) < 70_000 and not config["acceleration"]["quantization"]:
    # bf16 的 22B 模型（44 GB）放不進 40GB 的卡：底模量化成 INT8（約 22 GB），LoRA 照常訓練
    config["acceleration"]["quantization"] = "int8-quanto"
    print("40GB 的卡：底模用 INT8 量化訓練，每步比 80GB 慢一些")
done = sorted(glob.glob(f"{RUN_DIR}/checkpoints/*step_*.safetensors"))
config["model"]["load_checkpoint"] = f"{RUN_DIR}/checkpoints" if done else None
print(f"接著 {os.path.basename(done[-1])} 訓練" if done else "從頭開始訓練")
with open(f"{LOCAL}/train.yaml", "w") as f:
    yaml.safe_dump(config, f, sort_keys=False, allow_unicode=True)
sh(["uv", "run", "python", "scripts/train.py", f"{LOCAL}/train.yaml"], cwd=TRAINER)

## 結果

LoRA 在 Drive 的 `reveil/runs/<RUN_NAME>/checkpoints/lora_weights_step_XXXXX.safetensors`。

- 用 `SMOKE_TEST = True` 跑通之後，改回 `False` 正式訓練。
- 配對在 `reveil/pairs/<DATA_MODE>_<KIND>/`，前處理資料在 `reveil/precomputed/<DATA_MODE>_<KIND>/`（`KIND` 是 `smoke` 或 `main`）。改了 `CAPTION`：刪掉目前模式的這兩個資料夾再重跑。
- 真實配對的 `reveil/align/` 和 `reveil/regions/real/` 由冒煙測試與正式訓練共用，重新連線後會沿用。改了 `TARGET`、`COLOUR_FIT` 或視窗上限，要先刪掉目前模式的配對和前處理資料夾，再重建資料；只改 `RESOLUTIONS` 則刪掉目前模式的前處理資料夾，以 `REBUILD_DATA = True` 重跑。
- 推論（把 LoRA 套到 distilled 模型上還原影片）是 Reveil 的下一步。